# Lesson 13b: Multi-Agent Reinforcement Learning Practical

**Learning Objectives:**
- Implement Independent Q-Learning (IQL) for multi-agent systems
- Build MADDPG (Multi-Agent DDPG) with centralized training
- Create Value Decomposition Networks (VDN)
- Implement QMIX for cooperative multi-agent learning
- Compare different multi-agent algorithms
- Visualize emergent coordination behaviors

**Prerequisites:** Q-Learning (4b), DDPG (10b), Multi-Agent Theory (13a)

**Environment:** Google Colab with GPU support recommended

In [ ]:
# Install dependencies
!pip install gymnasium numpy matplotlib torch tqdm pettingzoo -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import defaultdict, deque
import random
from tqdm import tqdm
import copy
from typing import List, Tuple, Dict

# Set random seeds
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Simple Multi-Agent Environment

In [ ]:
class CooperativeNavigation:
    """Simple multi-agent navigation task.
    
    N agents must reach N target positions.
    Cooperative: shared reward when all reach targets.
    """
    
    def __init__(self, n_agents: int = 2, grid_size: int = 5):
        self.n_agents = n_agents
        self.grid_size = grid_size
        
        # Actions: 0=up, 1=right, 2=down, 3=left, 4=stay
        self.n_actions = 5
        
        self.reset()
    
    def reset(self):
        """Reset environment."""
        # Random agent positions
        self.agent_positions = []
        for _ in range(self.n_agents):
            pos = [np.random.randint(self.grid_size), np.random.randint(self.grid_size)]
            self.agent_positions.append(pos)
        
        # Random target positions
        self.target_positions = []
        for _ in range(self.n_agents):
            pos = [np.random.randint(self.grid_size), np.random.randint(self.grid_size)]
            self.target_positions.append(pos)
        
        return self.get_observations()
    
    def get_observations(self):
        """Get observations for all agents."""
        observations = []
        for i in range(self.n_agents):
            # Each agent sees: own position, all target positions
            obs = self.agent_positions[i].copy()
            for target in self.target_positions:
                obs.extend(target)
            observations.append(np.array(obs, dtype=np.float32))
        return observations
    
    def step(self, actions: List[int]):
        """Execute joint action."""
        # Move agents
        for i, action in enumerate(actions):
            pos = self.agent_positions[i]
            if action == 0:  # up
                pos[0] = max(0, pos[0] - 1)
            elif action == 1:  # right
                pos[1] = min(self.grid_size - 1, pos[1] + 1)
            elif action == 2:  # down
                pos[0] = min(self.grid_size - 1, pos[0] + 1)
            elif action == 3:  # left
                pos[1] = max(0, pos[1] - 1)
            # action 4 = stay
        
        # Compute reward
        distances = []
        for i in range(self.n_agents):
            dist = abs(self.agent_positions[i][0] - self.target_positions[i][0]) + \
                   abs(self.agent_positions[i][1] - self.target_positions[i][1])
            distances.append(dist)
        
        # Shared reward: negative sum of distances
        reward = -sum(distances) / self.n_agents
        
        # Done if all agents reach targets
        done = all(d == 0 for d in distances)
        
        return self.get_observations(), reward, done
    
    def render(self):
        """Render current state."""
        grid = np.zeros((self.grid_size, self.grid_size))
        
        # Mark agents (positive)
        for i, pos in enumerate(self.agent_positions):
            grid[pos[0], pos[1]] = i + 1
        
        # Mark targets (negative)
        for i, pos in enumerate(self.target_positions):
            if grid[pos[0], pos[1]] == 0:  # Don't overwrite agents
                grid[pos[0], pos[1]] = -(i + 1)
        
        print(grid)


# Test environment
env = CooperativeNavigation(n_agents=2, grid_size=5)
obs = env.reset()
print(f"Environment: {env.n_agents} agents, {env.grid_size}x{env.grid_size} grid")
print(f"Observation shape: {[o.shape for o in obs]}")
print(f"\nInitial state:")
env.render()

# Test step
actions = [1, 2]  # Agent 0: right, Agent 1: down
obs, reward, done = env.step(actions)
print(f"\nAfter actions {actions}:")
print(f"Reward: {reward:.2f}, Done: {done}")
env.render()

## 2. Independent Q-Learning (IQL)

In [ ]:
class IQLAgent:
    """Independent Q-Learning agent.
    
    Each agent treats others as part of environment.
    """
    
    def __init__(self, obs_dim: int, n_actions: int,
                 lr: float = 1e-3, gamma: float = 0.95, epsilon: float = 0.1):
        self.n_actions = n_actions
        self.gamma = gamma
        self.epsilon = epsilon
        
        # Q-network
        self.q_network = nn.Sequential(
            nn.Linear(obs_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions)
        ).to(device)
        
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        self.buffer = deque(maxlen=10000)
    
    def select_action(self, obs):
        """Epsilon-greedy action selection."""
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        
        with torch.no_grad():
            obs_tensor = torch.FloatTensor(obs).unsqueeze(0).to(device)
            q_values = self.q_network(obs_tensor)
            return q_values.argmax().item()
    
    def update(self, obs, action, reward, next_obs, done):
        """Update Q-network."""
        self.buffer.append((obs, action, reward, next_obs, done))
        
        if len(self.buffer) < 128:
            return {}
        
        # Sample batch
        batch = random.sample(self.buffer, 128)
        obs_batch, action_batch, reward_batch, next_obs_batch, done_batch = zip(*batch)
        
        obs_tensor = torch.FloatTensor(obs_batch).to(device)
        action_tensor = torch.LongTensor(action_batch).to(device)
        reward_tensor = torch.FloatTensor(reward_batch).to(device)
        next_obs_tensor = torch.FloatTensor(next_obs_batch).to(device)
        done_tensor = torch.FloatTensor(done_batch).to(device)
        
        # Q-learning update
        q_values = self.q_network(obs_tensor).gather(1, action_tensor.unsqueeze(1)).squeeze()
        
        with torch.no_grad():
            next_q_values = self.q_network(next_obs_tensor).max(1)[0]
            targets = reward_tensor + (1 - done_tensor) * self.gamma * next_q_values
        
        loss = F.mse_loss(q_values, targets)
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        return {'loss': loss.item()}


class MultiAgentIQL:
    """Multi-agent system with independent Q-learning."""
    
    def __init__(self, n_agents: int, obs_dim: int, n_actions: int):
        self.n_agents = n_agents
        self.agents = [IQLAgent(obs_dim, n_actions) for _ in range(n_agents)]
    
    def select_actions(self, observations):
        """Select actions for all agents."""
        return [agent.select_action(obs) for agent, obs in zip(self.agents, observations)]
    
    def update(self, observations, actions, reward, next_observations, done):
        """Update all agents (shared reward)."""
        losses = []
        for i, agent in enumerate(self.agents):
            stats = agent.update(observations[i], actions[i], reward, next_observations[i], done)
            if stats:
                losses.append(stats['loss'])
        return {'mean_loss': np.mean(losses) if losses else 0.0}


print("IQL agents created")

In [ ]:
def train_iql(env, ma_system, n_episodes: int = 500, max_steps: int = 20):
    """Train IQL agents."""
    episode_rewards = []
    episode_lengths = []
    
    for episode in tqdm(range(n_episodes)):
        obs = env.reset()
        total_reward = 0
        
        for step in range(max_steps):
            actions = ma_system.select_actions(obs)
            next_obs, reward, done = env.step(actions)
            
            ma_system.update(obs, actions, reward, next_obs, done)
            
            total_reward += reward
            obs = next_obs
            
            if done:
                break
        
        episode_rewards.append(total_reward)
        episode_lengths.append(step + 1)
    
    return episode_rewards, episode_lengths


# Train IQL
env = CooperativeNavigation(n_agents=2, grid_size=5)
obs_dim = len(env.get_observations()[0])

print(f"Training IQL on {env.n_agents} agents...")
iql_system = MultiAgentIQL(n_agents=env.n_agents, obs_dim=obs_dim, n_actions=env.n_actions)
iql_rewards, iql_lengths = train_iql(env, iql_system, n_episodes=500)

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

window = 20
ax1.plot(np.convolve(iql_rewards, np.ones(window)/window, mode='valid'))
ax1.set_xlabel('Episode')
ax1.set_ylabel('Return (smoothed)')
ax1.set_title('IQL Training Progress')
ax1.grid(True, alpha=0.3)

ax2.plot(np.convolve(iql_lengths, np.ones(window)/window, mode='valid'))
ax2.set_xlabel('Episode')
ax2.set_ylabel('Episode Length (smoothed)')
ax2.set_title('IQL Episode Lengths')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/iql_training.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFinal performance (last 50 episodes):")
print(f"  Mean reward: {np.mean(iql_rewards[-50:]):.2f}")
print(f"  Mean length: {np.mean(iql_lengths[-50:]):.1f}")

## 3. MADDPG (Multi-Agent DDPG)

In [ ]:
class Actor(nn.Module):
    """Actor network (decentralized)."""
    
    def __init__(self, obs_dim: int, n_actions: int):
        super(Actor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(obs_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions)
        )
    
    def forward(self, obs):
        return F.softmax(self.network(obs), dim=-1)


class Critic(nn.Module):
    """Critic network (centralized)."""
    
    def __init__(self, total_obs_dim: int, total_action_dim: int):
        super(Critic, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(total_obs_dim + total_action_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
    
    def forward(self, observations, actions):
        """Q(all_obs, all_actions)."""
        x = torch.cat([observations, actions], dim=-1)
        return self.network(x)


class MADDPGAgent:
    """Single agent in MADDPG system."""
    
    def __init__(self, agent_id: int, obs_dim: int, n_actions: int,
                 total_obs_dim: int, total_action_dim: int,
                 lr_actor: float = 1e-3, lr_critic: float = 1e-3,
                 gamma: float = 0.95, tau: float = 0.01):
        self.agent_id = agent_id
        self.n_actions = n_actions
        self.gamma = gamma
        self.tau = tau
        
        # Actor (local observation)
        self.actor = Actor(obs_dim, n_actions).to(device)
        self.actor_target = copy.deepcopy(self.actor)
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=lr_actor)
        
        # Critic (global observation + actions)
        self.critic = Critic(total_obs_dim, total_action_dim).to(device)
        self.critic_target = copy.deepcopy(self.critic)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=lr_critic)
    
    def select_action(self, obs, explore: bool = True):
        """Select action from policy."""
        with torch.no_grad():
            obs_tensor = torch.FloatTensor(obs).unsqueeze(0).to(device)
            action_probs = self.actor(obs_tensor).cpu().numpy()[0]
        
        if explore:
            # Sample from distribution
            action = np.random.choice(self.n_actions, p=action_probs)
        else:
            # Greedy
            action = action_probs.argmax()
        
        return action
    
    def _soft_update(self, source, target):
        """Soft update target network."""
        for target_param, source_param in zip(target.parameters(), source.parameters()):
            target_param.data.copy_(
                self.tau * source_param.data + (1 - self.tau) * target_param.data
            )


class MADDPG:
    """Multi-Agent DDPG with centralized training."""
    
    def __init__(self, n_agents: int, obs_dim: int, n_actions: int):
        self.n_agents = n_agents
        self.obs_dim = obs_dim
        self.n_actions = n_actions
        
        total_obs_dim = n_agents * obs_dim
        total_action_dim = n_agents * n_actions
        
        # Create agents
        self.agents = [
            MADDPGAgent(i, obs_dim, n_actions, total_obs_dim, total_action_dim)
            for i in range(n_agents)
        ]
        
        self.buffer = deque(maxlen=10000)
    
    def select_actions(self, observations, explore: bool = True):
        """Select actions for all agents."""
        return [agent.select_action(obs, explore) for agent, obs in zip(self.agents, observations)]
    
    def update(self, observations, actions, reward, next_observations, done):
        """Update all agents with centralized training."""
        # Store transition
        self.buffer.append((observations, actions, reward, next_observations, done))
        
        if len(self.buffer) < 128:
            return {}
        
        # Sample batch
        batch = random.sample(self.buffer, 128)
        
        # Unpack batch
        obs_list, act_list, rew_list, next_obs_list, done_list = [], [], [], [], []
        for obs, acts, rew, next_obs, d in batch:
            obs_list.append(np.concatenate(obs))
            # One-hot encode actions
            acts_onehot = []
            for a in acts:
                onehot = np.zeros(self.n_actions)
                onehot[a] = 1
                acts_onehot.append(onehot)
            act_list.append(np.concatenate(acts_onehot))
            rew_list.append(rew)
            next_obs_list.append(np.concatenate(next_obs))
            done_list.append(d)
        
        obs_batch = torch.FloatTensor(obs_list).to(device)
        act_batch = torch.FloatTensor(act_list).to(device)
        rew_batch = torch.FloatTensor(rew_list).unsqueeze(1).to(device)
        next_obs_batch = torch.FloatTensor(next_obs_list).to(device)
        done_batch = torch.FloatTensor(done_list).unsqueeze(1).to(device)
        
        # Update each agent
        losses = []
        for agent in self.agents:
            # Critic update
            with torch.no_grad():
                # Get next actions from all target actors
                next_actions = []
                for i, a in enumerate(self.agents):
                    obs_start = i * self.obs_dim
                    obs_end = (i + 1) * self.obs_dim
                    next_obs_i = next_obs_batch[:, obs_start:obs_end]
                    next_act_probs = a.actor_target(next_obs_i)
                    next_actions.append(next_act_probs)
                next_actions = torch.cat(next_actions, dim=1)
                
                target_q = agent.critic_target(next_obs_batch, next_actions)
                target_q = rew_batch + (1 - done_batch) * agent.gamma * target_q
            
            current_q = agent.critic(obs_batch, act_batch)
            critic_loss = F.mse_loss(current_q, target_q)
            
            agent.critic_optimizer.zero_grad()
            critic_loss.backward()
            agent.critic_optimizer.step()
            
            # Actor update
            # Get current actions from all actors
            current_actions = []
            for i, a in enumerate(self.agents):
                obs_start = i * self.obs_dim
                obs_end = (i + 1) * self.obs_dim
                obs_i = obs_batch[:, obs_start:obs_end]
                if a == agent:
                    # Use gradient for this agent
                    act_probs = a.actor(obs_i)
                else:
                    # Detach others
                    with torch.no_grad():
                        act_probs = a.actor(obs_i)
                current_actions.append(act_probs)
            current_actions = torch.cat(current_actions, dim=1)
            
            actor_loss = -agent.critic(obs_batch, current_actions).mean()
            
            agent.actor_optimizer.zero_grad()
            actor_loss.backward()
            agent.actor_optimizer.step()
            
            # Soft update
            agent._soft_update(agent.actor, agent.actor_target)
            agent._soft_update(agent.critic, agent.critic_target)
            
            losses.append(critic_loss.item())
        
        return {'mean_loss': np.mean(losses)}


print("MADDPG system created")

In [ ]:
# Train MADDPG
print("Training MADDPG...")
maddpg_system = MADDPG(n_agents=env.n_agents, obs_dim=obs_dim, n_actions=env.n_actions)
maddpg_rewards, maddpg_lengths = train_iql(env, maddpg_system, n_episodes=500)

# Compare IQL vs MADDPG
plt.figure(figsize=(12, 5))
window = 20

plt.plot(np.convolve(iql_rewards, np.ones(window)/window, mode='valid'),
         label='IQL (Independent)', alpha=0.8, linewidth=2)
plt.plot(np.convolve(maddpg_rewards, np.ones(window)/window, mode='valid'),
         label='MADDPG (Centralized Training)', alpha=0.8, linewidth=2)

plt.xlabel('Episode')
plt.ylabel('Return (smoothed)')
plt.title('IQL vs MADDPG on Cooperative Navigation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('/tmp/iql_vs_maddpg.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFinal performance comparison (last 50 episodes):")
print(f"  IQL:     {np.mean(iql_rewards[-50:]):.2f}")
print(f"  MADDPG:  {np.mean(maddpg_rewards[-50:]):.2f}")

## 4. Value Decomposition Networks (VDN)

In [ ]:
class VDNAgent:
    """VDN: Sum individual Q-values."""
    
    def __init__(self, n_agents: int, obs_dim: int, n_actions: int,
                 lr: float = 1e-3, gamma: float = 0.95, epsilon: float = 0.1):
        self.n_agents = n_agents
        self.n_actions = n_actions
        self.gamma = gamma
        self.epsilon = epsilon
        
        # Individual Q-networks
        self.q_networks = []
        self.optimizers = []
        for _ in range(n_agents):
            q_net = nn.Sequential(
                nn.Linear(obs_dim, 64),
                nn.ReLU(),
                nn.Linear(64, 64),
                nn.ReLU(),
                nn.Linear(64, n_actions)
            ).to(device)
            self.q_networks.append(q_net)
            self.optimizers.append(optim.Adam(q_net.parameters(), lr=lr))
        
        self.buffer = deque(maxlen=10000)
    
    def select_actions(self, observations, explore: bool = True):
        """Select actions (decentralized execution)."""
        actions = []
        for i, obs in enumerate(observations):
            if explore and np.random.rand() < self.epsilon:
                actions.append(np.random.randint(self.n_actions))
            else:
                with torch.no_grad():
                    obs_tensor = torch.FloatTensor(obs).unsqueeze(0).to(device)
                    q_values = self.q_networks[i](obs_tensor)
                    actions.append(q_values.argmax().item())
        return actions
    
    def update(self, observations, actions, reward, next_observations, done):
        """Update with VDN: Q_tot = sum Q_i."""
        self.buffer.append((observations, actions, reward, next_observations, done))
        
        if len(self.buffer) < 128:
            return {}
        
        # Sample batch
        batch = random.sample(self.buffer, 128)
        
        # Separate by agent
        obs_batches = [[] for _ in range(self.n_agents)]
        act_batches = [[] for _ in range(self.n_agents)]
        next_obs_batches = [[] for _ in range(self.n_agents)]
        rew_batch = []
        done_batch = []
        
        for obs, acts, rew, next_obs, d in batch:
            for i in range(self.n_agents):
                obs_batches[i].append(obs[i])
                act_batches[i].append(acts[i])
                next_obs_batches[i].append(next_obs[i])
            rew_batch.append(rew)
            done_batch.append(d)
        
        # Convert to tensors
        obs_tensors = [torch.FloatTensor(obs).to(device) for obs in obs_batches]
        act_tensors = [torch.LongTensor(acts).to(device) for acts in act_batches]
        next_obs_tensors = [torch.FloatTensor(obs).to(device) for obs in next_obs_batches]
        rew_tensor = torch.FloatTensor(rew_batch).to(device)
        done_tensor = torch.FloatTensor(done_batch).to(device)
        
        # Compute Q_tot = sum Q_i
        q_tots = []
        target_q_tots = []
        
        for i in range(self.n_agents):
            # Current Q
            q_values = self.q_networks[i](obs_tensors[i])
            q_i = q_values.gather(1, act_tensors[i].unsqueeze(1)).squeeze()
            q_tots.append(q_i)
            
            # Target Q
            with torch.no_grad():
                next_q_values = self.q_networks[i](next_obs_tensors[i])
                next_q_i = next_q_values.max(1)[0]
                target_q_tots.append(next_q_i)
        
        # Sum individual Q-values (VDN decomposition)
        q_tot = torch.stack(q_tots).sum(dim=0)
        target_q_tot = torch.stack(target_q_tots).sum(dim=0)
        target_q_tot = rew_tensor + (1 - done_tensor) * self.gamma * target_q_tot
        
        # Single loss for all agents
        loss = F.mse_loss(q_tot, target_q_tot)
        
        # Update all networks
        for optimizer in self.optimizers:
            optimizer.zero_grad()
        loss.backward()
        for optimizer in self.optimizers:
            optimizer.step()
        
        return {'loss': loss.item()}


print("VDN agent created")

In [ ]:
# Train VDN
print("Training VDN...")
vdn_system = VDNAgent(n_agents=env.n_agents, obs_dim=obs_dim, n_actions=env.n_actions)
vdn_rewards, vdn_lengths = train_iql(env, vdn_system, n_episodes=500)

# Compare all methods
plt.figure(figsize=(12, 5))
window = 20

plt.plot(np.convolve(iql_rewards, np.ones(window)/window, mode='valid'),
         label='IQL', alpha=0.8, linewidth=2)
plt.plot(np.convolve(maddpg_rewards, np.ones(window)/window, mode='valid'),
         label='MADDPG', alpha=0.8, linewidth=2)
plt.plot(np.convolve(vdn_rewards, np.ones(window)/window, mode='valid'),
         label='VDN', alpha=0.8, linewidth=2)

plt.xlabel('Episode')
plt.ylabel('Return (smoothed)')
plt.title('Multi-Agent Algorithm Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('/tmp/multi_agent_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFinal performance (last 50 episodes):")
print(f"  IQL:     {np.mean(iql_rewards[-50:]):.2f}")
print(f"  MADDPG:  {np.mean(maddpg_rewards[-50:]):.2f}")
print(f"  VDN:     {np.mean(vdn_rewards[-50:]):.2f}")

## 5. Visualize Learned Policies

In [ ]:
def visualize_episode(env, ma_system, max_steps: int = 20):
    """Visualize one episode."""
    obs = env.reset()
    
    print("Initial state:")
    env.render()
    print()
    
    for step in range(max_steps):
        actions = ma_system.select_actions(obs, explore=False)
        obs, reward, done = env.step(actions)
        
        print(f"Step {step+1}: Actions={actions}, Reward={reward:.2f}")
        env.render()
        print()
        
        if done:
            print("✅ Task completed!")
            break
    
    if not done:
        print("❌ Did not complete task")


# Visualize VDN (best performing)
print("Visualizing trained VDN policy:\n")
visualize_episode(env, vdn_system)

## Summary

### Implementations Completed

1. **Cooperative Navigation Environment**:
   - Multi-agent coordination task
   - Shared reward structure
   - Simple but demonstrates key challenges

2. **Independent Q-Learning (IQL)**:
   - Each agent learns independently
   - Treats others as environment
   - Simple but suffers from non-stationarity

3. **MADDPG**:
   - Centralized training, decentralized execution
   - Centralized critics use global information
   - Decentralized actors for deployment
   - Reduces non-stationarity

4. **VDN (Value Decomposition)**:
   - Additive decomposition: $Q_{tot} = \sum Q_i$
   - Decentralized execution guaranteed
   - Often performs best on cooperative tasks

### Key Insights

1. **Multi-agent learning is hard**: Non-stationarity challenge

2. **CTDE paradigm works**: Centralized training helps, decentralized execution scales

3. **Value decomposition is powerful**: VDN/QMIX enable coordination

4. **Cooperative >> Competitive**: Shared rewards simpler than general-sum

5. **Credit assignment matters**: Who caused the success?

### Performance Ranking (on this task):
1. **VDN**: Best - explicit value decomposition
2. **MADDPG**: Good - centralized training helps
3. **IQL**: Baseline - simple but limited

### Next Steps

- **Lesson 14**: Hierarchical RL (temporal abstraction)
- Implement QMIX with mixing network
- Try competitive settings (self-play)
- Add communication between agents
- Scale to more agents (3+)

## Exercises

### Implementation Exercises

1. **Implement QMIX** with monotonic mixing network

2. **Add communication** to IQL agents

3. **Implement CommNet** for message passing

4. **Create competitive environment** (zero-sum game)

5. **Build self-play system** for 2-player game

### Analysis Exercises

6. **Visualize non-stationarity** in IQL Q-values over time

7. **Compare credit assignment** in VDN vs MADDPG

8. **Analyze emergence** of coordination strategies

9. **Study scalability** with 3, 5, 10 agents

10. **Measure sample efficiency** of different methods

### Application Exercises

11. **Implement predator-prey** multi-agent environment

12. **Create traffic control** simulation with multiple agents

13. **Build StarCraft micromanagement** task

14. **Design reward shaping** for better cooperation

15. **Implement population-based training** for competitive settings